In [10]:
import numpy as np
import pandas as pd
import matplotlib as mlt
import re

In [ ]:
def clean_text(text):
    """Normalizes text by lowercasing, stripping punctuation, and expanding abbreviations."""
    if pd.isna(text):
        return ""
    
    # Lowercase everything
    text = str(text).lower()
    
    # Strip accents (useful for French test data)
    text = text.encode('ascii', 'ignore').decode('utf-8')
    
    # Remove punctuation, keeping only alphanumeric characters and spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Standardize common legal and address abbreviations using word boundaries (\b)
    replacements = {
        r'\bcorp\b': 'corporation',
        r'\binc\b': 'incorporated',
        r'\bltd\b': 'limited',
        r'\bpvt\b': 'private',
        r'\bst\b': 'street',
        r'\brd\b': 'road',
        r'\bave\b': 'avenue',
        r'\bdr\b': 'drive',
        r'\bblvd\b': 'boulevard'
    }
    
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
        
    # Collapse multiple spaces into a single space and strip leading/trailing whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_and_preprocess(filepath):
    """Loads a TSV file and creates a unified blocking key."""
    # Reading with sep="\t" is mandatory as ID lists and addresses contain commas[cite: 1]
    df = pd.read_csv(filepath, sep="\t")
    
    # Apply cleaning to names and addresses
    df['clean_name'] = df['business_name'].apply(clean_text)
    df['clean_address'] = df['business_address'].apply(clean_text)
    
    # Create a single combined string to use for the TF-IDF blocking stage
    df['blocking_key'] = df['clean_name'] + " " + df['clean_address']
    
    return df

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

def generate_candidates(s1_df, s2_df, s3_df, top_k=15):
    """
    Generates top K candidate matches from Source 2 and Source 3 for each Source 1 entity.
    Returns a DataFrame formatted for candidate_pairs.tsv.
    """
    # Combine Source 2 and Source 3 into a single search pool
    s23_df = pd.concat([s2_df, s3_df], ignore_index=True)
    
    # Fit the vectorizer on all available text to build a shared vocabulary
    all_text = pd.concat([s1_df['blocking_key'], s23_df['blocking_key']])
    
    # Use character n-grams (3 to 5 characters) to capture partial words and typos
    print("Building TF-IDF Vocabulary...")
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2)
    vectorizer.fit(all_text)
    
    # Convert text strings into sparse numerical vectors
    print("Vectorizing strings...")
    s1_vectors = vectorizer.transform(s1_df['blocking_key'])
    s23_vectors = vectorizer.transform(s23_df['blocking_key'])
    
    # Build the Nearest Neighbors search index using cosine distance
    # n_jobs=-1 uses all available CPU cores for speed
    print("Building Search Index...")
    nn = NearestNeighbors(n_neighbors=top_k, metric='cosine', n_jobs=-1)
    nn.fit(s23_vectors)
    
    # Query the index: Find the nearest S2/S3 vectors for every S1 vector
    print("Searching for candidates...")
    distances, indices = nn.kneighbors(s1_vectors)
    
    # Map the numerical indices back to the actual entity IDs
    s23_entity_ids = s23_df['entity_id'].values
    s1_entity_ids = s1_df['entity_id'].values
    
    candidates_data = []
    
    for i, s1_id in enumerate(s1_entity_ids):
        # Extract the matched IDs
        matched_ids = [s23_entity_ids[idx] for idx in indices[i]]
        
        # The challenge requires a comma-separated string of S2/S3 IDs[cite: 1]
        candidates_data.append({
            'source1_entity_id': s1_id,
            'candidate_entity_ids': ",".join(matched_ids)
        })
        
    candidate_df = pd.DataFrame(candidates_data)
    
    # Save the exact set fed into your matching model[cite: 1]
    # candidate_df.to_csv("output/candidate_pairs.tsv", sep="\t", index=False)
    
    return candidate_df

# Example usage (assuming s1_train, s2_train, s3_train are DataFrames from step 1):
# candidate_pairs_df = generate_candidates(s1_train, s2_train, s3_train, top_k=15)

In [12]:
from rapidfuzz import fuzz, distance

def extract_digits(text):
    """Extracts all numbers from a string (e.g., house numbers, postal codes)."""
    if pd.isna(text):
        return ""
    return "".join(re.findall(r'\d+', str(text)))

def create_feature_matrix(candidate_df, s1_df, s2_df, s3_df, ground_truth_df=None):
    """
    Computes pairwise similarity features for every candidate pair.
    Optionally joins ground truth labels for training.
    """
    # Create fast lookup dictionaries for entity attributes
    all_entities = pd.concat([s1_df, s2_df, s3_df], ignore_index=True)
    lookup = all_entities.set_index('entity_id').to_dict(orient='index')
    
    # Explode candidate list into individual (s1_id, candidate_id) pairs
    pairs = []
    for _, row in candidate_df.iterrows():
        s1_id = row['source1_entity_id']
        c_ids = row['candidate_entity_ids']
        if pd.isna(c_ids) or not str(c_ids).strip():
            continue
        for c_id in str(c_ids).split(','):
            c_id = c_id.strip()
            if c_id:
                pairs.append({'source1_entity_id': s1_id, 'candidate_entity_id': c_id})
                
    pair_df = pd.DataFrame(pairs)
    if pair_df.empty:
        return pair_df

    features = []
    
    for _, row in pair_df.iterrows():
        s1_id = row['source1_entity_id']
        cand_id = row['candidate_entity_id']
        
        rec1 = lookup.get(s1_id, {})
        rec2 = lookup.get(cand_id, {})
        
        # Strings for name comparison
        name1 = rec1.get('clean_name', '')
        name2 = rec2.get('clean_name', '')
        
        # Strings for address comparison
        addr1 = rec1.get('clean_address', '')
        addr2 = rec2.get('clean_address', '')
        
        # Extract digits from addresses (critical for street numbers and PIN codes)
        num1 = extract_digits(addr1)
        num2 = extract_digits(addr2)
        
        feat = {
            'source1_entity_id': s1_id,
            'candidate_entity_id': cand_id,
            
            # --- Business Name Similarities ---
            'name_ratio': fuzz.ratio(name1, name2) / 100.0,
            'name_partial_ratio': fuzz.partial_ratio(name1, name2) / 100.0,
            'name_token_sort_ratio': fuzz.token_sort_ratio(name1, name2) / 100.0,
            'name_token_set_ratio': fuzz.token_set_ratio(name1, name2) / 100.0,
            'name_jaro_winkler': distance.JaroWinkler.similarity(name1, name2),
            
            # --- Business Address Similarities ---
            'addr_ratio': fuzz.ratio(addr1, addr2) / 100.0,
            'addr_partial_ratio': fuzz.partial_ratio(addr1, addr2) / 100.0,
            'addr_token_sort_ratio': fuzz.token_sort_ratio(addr1, addr2) / 100.0,
            'addr_token_set_ratio': fuzz.token_set_ratio(addr1, addr2) / 100.0,
            'addr_jaro_winkler': distance.JaroWinkler.similarity(addr1, addr2),
            
            # --- Structural / Binary Features ---
            'country_match': 1.0 if rec1.get('country') == rec2.get('country') else 0.0,
            'exact_number_match': 1.0 if (num1 and num2 and num1 == num2) else 0.0,
            'has_number_overlap': 1.0 if (num1 and num2 and (num1 in num2 or num2 in num1)) else 0.0,
        }
        features.append(feat)
        
    feat_df = pd.DataFrame(features)
    
    # Attach labels if ground truth is provided (Training mode)
    if ground_truth_df is not None:
        # Build lookup set of true pairs: (s1_id, matched_id)
        gt_map = set()
        for _, row in ground_truth_df.iterrows():
            s1_id = row['source1_entity_id']
            m_ids = row['matched_entity_ids']
            if pd.notna(m_ids) and str(m_ids).strip():
                for m_id in str(m_ids).split(','):
                    gt_map.add((s1_id, m_id.strip()))
                    
        feat_df['is_match'] = feat_df.apply(
            lambda r: 1 if (r['source1_entity_id'], r['candidate_entity_id']) in gt_map else 0, 
            axis=1
        )
        
    return feat_df

# Example usage:
# train_features = create_feature_matrix(candidate_pairs_df, s1_train, s2_train, s3_train

In [ ]:
import xgboost as xgb
from sklearn.metrics import precision_score, recall_score, fbeta_score

def train_and_tune_model(features_df):
    """
    Trains an XGBoost classifier on candidate features and finds the optimal 
    probability threshold to maximize the F_0.5 score.
    """
    # Separate features and target label
    feature_cols = [col for col in features_df.columns if col not in ['source1_entity_id', 'candidate_entity_id', 'is_match']]
    
    X = features_df[feature_cols]
    y = features_df['is_match']
    
    # Initialize XGBoost Classifier
    # scale_pos_weight is useful if the dataset is highly imbalanced (many more 0s than 1s)
    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
    
    print("Training XGBoost Classifier...")
    model.fit(X, y)
    
    # Predict probabilities (we want the probability of class 1: match)
    probabilities = model.predict_proba(X)[:, 1]
    features_df['match_prob'] = probabilities
    
    # Tune the threshold for the F_0.5 metric
    print("Tuning decision threshold for F_0.5 metric...")
    best_threshold = 0.5
    best_f05 = 0.0
    
    # Iterate through possible thresholds to find the peak F_0.5 score
    thresholds = np.arange(0.3, 0.95, 0.01)
    for thresh in thresholds:
        # Convert probabilities to hard 1 or 0 based on current threshold
        y_pred = (probabilities >= thresh).astype(int)
        
        # Calculate F_0.5 score: beta=0.5 weighs precision 2x over recall
        f05 = fbeta_score(y, y_pred, beta=0.5, zero_division=0)
        
        if f05 > best_f05:
            best_f05 = f05
            best_threshold = thresh
            
    print(f"Optimal Threshold Found: {best_threshold:.2f}")
    print(f"Best Training F_0.5 Score: {best_f05:.4f}")
    
    return model, best_threshold

def generate_final_submission(model, test_features_df, threshold, top_n_matches=3):
    """
    Applies the trained model and threshold to test data, formatting the output
    exactly as required for matching_results.tsv.
    """
    feature_cols = [col for col in test_features_df.columns if col not in ['source1_entity_id', 'candidate_entity_id']]
    X_test = test_features_df[feature_cols]
    
    # Predict probabilities on test features
    test_features_df['match_prob'] = model.predict_proba(X_test)[:, 1]
    
    # Filter candidates passing the strict precision threshold
    confident_matches = test_features_df[test_features_df['match_prob'] >= threshold].copy()
    
    # Sort by probability so the highest confidence matches appear first
    confident_matches.sort_values(by=['source1_entity_id', 'match_prob'], ascending=[True, False], inplace=True)
    
    # Aggregate matches for each Source 1 ID
    # The challenge rules dictate comma-separated lists of matched S2/S3 IDs
    submission_data = []
    
    # Get all unique Source 1 IDs from the original test set to ensure no entities are missed
    all_s1_ids = test_features_df['source1_entity_id'].unique()
    
    for s1_id in all_s1_ids:
        # Get the confident matches for this specific Source 1 ID
        matches = confident_matches[confident_matches['source1_entity_id'] == s1_id]
        
        if matches.empty:
            # Leave matched_entity_ids empty for singletons (worth 1.0 if correctly predicted)
            matched_str = ""
        else:
            # Join top N confident matches into a comma-separated string
            matched_list = matches['candidate_entity_id'].head(top_n_matches).tolist()
            matched_str = ",".join(matched_list)
            
        submission_data.append({
            'source1_entity_id': s1_id,
            'matched_entity_ids': matched_str
        })
        
    submission_df = pd.DataFrame(submission_data)
    
    # Save output precisely as requested: tab-separated, no quotes, no index[cite: 1]
    # submission_df.to_csv("output/matching_results.tsv", sep="\t", index=False)
    
    return submission_df

# Example usage:
# trained_model, opt_thresh = train_and_tune_model(train_features_df)
# final_predictions = generate_final_submission(trained_model, test_features_df, opt_thresh)